# 03 — Gap Model

Gap model гледа разстоянието между появяванията на всяко число: current gap, average gap и overdue/underdue сигнали.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Gap statistics


In [ ]:
num_cols = number_columns(draws)
records = []
for number in range(1, 50):
    hit_indices = draws.index[draws[num_cols].eq(number).any(axis=1)].tolist()
    if not hit_indices:
        records.append({"number": number, "appearances": 0, "current_gap": np.nan, "avg_gap": np.nan, "max_gap": np.nan})
        continue
    gaps = np.diff(hit_indices)
    current_gap = len(draws) - 1 - hit_indices[-1]
    records.append({
        "number": number,
        "appearances": len(hit_indices),
        "current_gap": current_gap,
        "avg_gap": float(np.mean(gaps)) if len(gaps) else np.nan,
        "max_gap": int(np.max(gaps)) if len(gaps) else np.nan,
    })
gap_df = pd.DataFrame(records)
gap_df["gap_ratio"] = gap_df["current_gap"] / gap_df["avg_gap"]
gap_df.sort_values("gap_ratio", ascending=False).head(15)


## Най-голям current gap


In [ ]:
overdue = gap_df.sort_values("current_gap", ascending=False).head(12)
display(overdue)
ax = overdue.set_index("number")["current_gap"].plot(kind="bar", figsize=(10, 4), title="Numbers with largest current gap")
ax.set_xlabel("Number")
ax.set_ylabel("Current gap")
plt.show()


## Current gap спрямо average gap


In [ ]:
plot_df = gap_df.dropna(subset=["avg_gap", "current_gap"]).set_index("number")[["current_gap", "avg_gap"]]
ax = plot_df.plot(kind="bar", figsize=(15, 5), title="Current gap vs average gap")
ax.set_xlabel("Number")
ax.set_ylabel("Gap")
plt.show()
